# Association of infant sleep rhythmicity by the gut microbiota diversity & volatility

Summary: Modulation of infant sleep by the gut microbiota: sleep rhythmicity (CFI).

Author, date: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from functions_script import load_tsv, scatterplot, plot_estimates_vertically

%matplotlib inline

In [ ]:
# paths
# IF ACCESS TO ALL STUDY DATA
path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
meta_data_path = f"{path}/data-sensitive/meta_data"
figures_path = f"{path}/figures"

# ELSE (ONLY ACCESS TO PUBLIC DATA)
# meta_data_path = "../data/meta_data"
# figures_path = "../figures"

In [ ]:
# create color palette
colors = [plt.cm.Spectral(i/float(6)) for i in range(7)]
timepoints_colors = [colors[0], colors[5], colors[6]]

## Load infants metadata

In [ ]:
# load infant metadata per age/timepoint
md_ages = load_tsv(f"{meta_data_path}/md_ages.tsv")
md_ages = md_ages.sort_values(by=["timepoint", "infant_id"])

print(md_ages.shape)
md_ages.head()

# Gut microbiota diversity & sleep rhythmicity

In [ ]:
# define data for diversity models
data_div = md_ages.dropna(subset=['observed_features', 'CFI']).copy()

In [ ]:
# impute missing ASQ values with medians
cols=["ASQ_Composite", "BCQ_Attunement", 'ASQ_Communication', 
      'ASQ_Finemotor', 'ASQ_Grossmotor', 'ASQ_Personalsocial', 
      'ASQ_Problemsolving']
data_div[cols] = data_div[cols].fillna(data_div[cols].median())

In [ ]:
# linear mixed effects model with observed features
model_of = smf.mixedlm("CFI ~ observed_features + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_div, 
                    groups=data_div["infant_id"]).fit()
print(model_of.summary())

In [ ]:
# linear mixed effects model with shannon entropy
model_sh = smf.mixedlm("CFI ~ shannon_entropy + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_div, 
                    groups=data_div["infant_id"]).fit()
print(model_sh.summary())

In [ ]:
# linear mixed effects model with pielou evenness
model_pi = smf.mixedlm("CFI ~ pielou_evenness + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_div, 
                    groups=data_div["infant_id"]).fit()
print(model_pi.summary())

In [ ]:
# linear mixed effects model with faith's phylogenetic diversity
model_fa = smf.mixedlm("CFI ~ faith_pd + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_div, 
                    groups=data_div["infant_id"]).fit()
print(model_fa.summary())

In [ ]:
# create scatterplots of significant alpha diversity and parenting vs CFI
fig, axs = plt.subplots(1, 3, figsize=(12, 6))

scatterplot(data_div, "observed_features", "CFI", "timepoint", 
                    "", "Sleep rhythmicity: circadian function index (CFI)", 
                    "Alpha diversity: \n Observed features",
                    timepoints_colors, ax=axs[0], loc_position = 'lower right')

scatterplot(data_div, "faith_pd", "CFI", "timepoint", 
                    "", "", "Alpha diversity: \n Faith phylogenetic diversity",
                    timepoints_colors, ax=axs[1], loc_position=None)

scatterplot(data_div, "BCQ_Attunement", "CFI", "timepoint", 
                    "", "", "Parenting style: \n BCQ Attuned care score",
                    timepoints_colors, ax=axs[2], loc_position = None)

plt.tight_layout()

plt.savefig(f"{figures_path}/cfi-and-sign.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# scatterplots of non-significant alpha diversity vs CFI
fig, axs = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

scatterplot(data_div, "shannon_entropy", "CFI", "timepoint", 
                    "", "Sleep rhythmicity: circadian function index (CFI)", "Alpha diversity: Shannon entropy",
                    timepoints_colors, axs[0])

scatterplot(data_div, "pielou_evenness", "CFI", "timepoint", 
                    "", "", "Alpha diversity: Pielou evenness",
                    timepoints_colors, axs[1], loc_position=None)

plt.tight_layout()

plt.savefig(f"{figures_path}/cfi-and-diversity-nonsign.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots for significant models
models = [model_of, model_fa]
titles = ["Sleep rhythmicity: circadian function index (CFI)", ""]
plot_estimates_vertically(models, titles, f"{figures_path}/sleep_rhythmicity_and_div_estimates_sign.pdf")

In [ ]:
# coefficient plots for non-significant models
models = [model_sh, model_pi]
plot_estimates_vertically(models, titles, f"{figures_path}/sleep_rhythmicity_and_div_estimates_nonsign.pdf")

# Gut microbiota volatility & sleep rhythmicity

In [ ]:
# define data for volatility models
data_volatility = md_ages.dropna(subset=['volatility_braycurtis', 'CFI']).copy()

In [ ]:
# fill missing parenting and ASQ data with median values
cols=["BCQ_Attunement", 'ASQ_Composite']
data_volatility[cols] = data_volatility[cols].fillna(data_volatility[cols].median())

In [ ]:
# linear mixed effects model with bray-curtis volatility
model_bc = smf.mixedlm("CFI ~ volatility_braycurtis + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_volatility, 
                    groups=data_volatility["infant_id"]).fit()
print(model_bc.summary())

In [ ]:
# linear mixed effects model with jaccard volatility
model_ja = smf.mixedlm("CFI ~ volatility_jaccard + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_volatility, 
                    groups=data_volatility["infant_id"]).fit()
print(model_ja.summary())

In [ ]:
# linear mixed effects model with unweighted unifrac volatility
model_uu = smf.mixedlm("CFI ~ volatility_unweighted_unifrac + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_volatility, 
                    groups=data_volatility["infant_id"]).fit()
print(model_uu.summary())

In [ ]:
# linear mixed effects model with weighted unifrac volatility
model_wu = smf.mixedlm("CFI ~ volatility_weighted_unifrac + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_volatility, 
                    groups=data_volatility["infant_id"]).fit()
print(model_wu.summary())

In [ ]:
# scatterplots of volatility vs CFI
fig, axs = plt.subplots(1, 4, figsize=(16, 6), sharey=True)

scatterplot(data_volatility, "volatility_braycurtis", "CFI", "timepoint", 
                    "", "Sleep rhythmicity: circadian function index (CFI)", "Temporal volatility: \n Bray-Curtis dissimilarity",
                    timepoints_colors, axs[0], loc_position = 'lower left')

scatterplot(data_volatility, "volatility_jaccard", "CFI", "timepoint", 
                    "", "", "Temporal volatility: \n Jaccard similarity index",
                    timepoints_colors, axs[1], loc_position = None)

scatterplot(data_volatility, "volatility_unweighted_unifrac", "CFI", "timepoint", 
                    "", "", "Temporal volatility: \n Unweighted UniFrac distance",
                    timepoints_colors, axs[2], loc_position = None)

scatterplot(data_volatility, "volatility_weighted_unifrac", "CFI", "timepoint", 
                    "", "", "Temporal volatility: \n Weighted UniFrac distance",
                    timepoints_colors, axs[3], loc_position = None)

plt.tight_layout()

plt.savefig(f"{figures_path}/cfi-and-volatility.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots for volatility models
models = [model_bc, model_ja, model_uu, model_wu]
titles = ["Sleep rhythmicity: circadian function index (CFI)", "", "", ""]
plot_estimates_vertically(models, titles, f"{figures_path}/sleep_rhythmicity_and_vol_estimates.pdf")